# RAG with LangChain and Hugging Face

This notebook implements a simple Retrieval-Augmented Generation (RAG) system using LangChain, Hugging Face embeddings, FAISS, and a Hugging Face question-answering model.

This fixed version avoids the Hugging Face dataset loader because it caused dependency conflicts in Colab.

## 1. Install required libraries

In [ ]:
!pip install -q transformers==4.41.2 tokenizers==0.19.1 huggingface-hub==0.23.4
!pip install -q sentence-transformers==3.0.1 faiss-cpu
!pip install -q langchain==0.2.6 langchain-community==0.2.6 langchain-huggingface==0.0.3 langchain-text-splitters==0.2.2

After running the install cell, restart the runtime once:

`Runtime` → `Restart runtime`

Then continue from the next cell.

## 2. Verify package versions

In [ ]:
import transformers
import langchain
import huggingface_hub

print("transformers:", transformers.__version__)
print("langchain:", langchain.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

## 3. Create sample documents

In [ ]:
from langchain_core.documents import Document

data = [
    Document(page_content="Cheesemaking is the process of producing cheese from milk. It usually involves coagulating milk, separating curds from whey, and aging or processing the curds."),
    Document(page_content="Milk can be coagulated using enzymes such as rennet or by adding acids. This causes proteins in the milk to form solid curds."),
    Document(page_content="During cheesemaking, curds are separated from whey. The curds are then salted, shaped, pressed, and sometimes aged to create different types of cheese."),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines information retrieval with language generation so the model can answer questions using external documents."),
    Document(page_content="FAISS is a library for efficient similarity search. In RAG systems, FAISS can store document embeddings and retrieve the most relevant chunks for a user query."),
    Document(page_content="Sentence transformers convert text into numerical vectors called embeddings. Similar texts usually have similar embeddings."),
]

print("Number of documents:", len(data))
print(data[0])

## 4. Split documents into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(data)

print("Number of chunks:", len(docs))
print(docs[0])

## 5. Create embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False}
)

test_embedding = embeddings.embed_query("This is a test document.")
print(test_embedding[:3])
print("Embedding size:", len(test_embedding))

## 6. Create FAISS vector store and retriever

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

retriever = db.as_retriever(search_kwargs={"k": 4})

print("Retriever created successfully")

## 7. Load Hugging Face question-answering model

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

print("QA pipeline loaded successfully")

## 8. Build the RAG function

In [ ]:
def rag_answer(question):
    retrieved_docs = retriever.invoke(question)

    context = " ".join([doc.page_content for doc in retrieved_docs])

    result = qa_pipeline(
        question=question,
        context=context
    )

    return {
        "question": question,
        "answer": result["answer"],
        "score": result["score"],
        "retrieved_context": context
    }

## 9. Test the RAG system

In [ ]:
question = "What is cheesemaking?"

result = rag_answer(question)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Score:", result["score"])
print("\nRetrieved context:")
print(result["retrieved_context"])

## 10. Try another question

In [ ]:
question = "What does RAG stand for?"

result = rag_answer(question)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Score:", result["score"])
print("\nRetrieved context:")
print(result["retrieved_context"])